In [ ]:
import pandas as pd
import requests
import datetime

"""
# Рекурсивно выводит все ключи вложенного словаря
def print_nested_keys(data, prefix=""):    
    if isinstance(data, dict):
        for key, value in data.items():
            print(f"{prefix}{key}")
            print_nested_keys(value, prefix + "  ")  # Уровень глубже
    elif isinstance(data, list):
        for i, item in enumerate(data):
            print(f"{prefix}[{i}]")
            print_nested_keys(item, prefix + "  ")
"""

#’https://lldev.thespacedevs.com/2.3.0/launches/' #для тестирования
#'https://ll.thespacedevs.com/2.3.0/launches/'
BASE_URL = 'https://lldev.thespacedevs.com/2.3.0/launches/'

# надо неделями грузить
ds = '2026-01-26' #datetime.datetime.now()
start_date = '2026-01-21' #'1980-01-01'
end_date = ds # datetime.datetime.now() 
limit = 100
# параметры для request API
params = {
    'window_start__gte': start_date,
    'window_start__lte': end_date,
    'mode': 'detailed',  # Для доступа к pad и country [web:90][web:92]
    'ordering': '-window_start',   # сортировка по дате (новые сверху)
    'limit': limit,                    # количество объектов (макс. 100 за раз)
    'format': 'json'
    }

#response.text (сырой HTML/JSON)
#response — объект requests.Response с сырыми данными
response = requests.get(BASE_URL, params=params, timeout=30)

if response.status_code != 200:
    print(f"Ошибка API: {response.status_code}")

#json() — автоматически:
#Читает response.content (байты)
#Декодирует UTF-8 → строку JSON
#Парсит JSON → Python объекты (dict, list, str, int)
data = response.json()

# безопасно извлекает список запусков из JSON API, если списка нет, подставит []
results = data.get('results', [])

#print_nested_keys(data)
#print(data.keys())         # dict_keys(['count', 'next', 'previous', 'results'])

# преобразует список вложенных JSON-словарей в плоский DataFrame(таблицу) Pandas.
df = pd.json_normalize(results)

# добавляет колонку с текущей меткой времени во все строки DataFrame.
df['update_at'] = datetime.datetime.now()

# показывает первые 5 строк DataFrame (для проверки данных).
df.head()

# Сохранение в csv
#df.to_csv("/home/coder/tmp/api__shushiato.csv", sep=";", header=True, index=False)

# Сохранение в Parquet
# pip install pyarrow --break-system-packages

output_path = "/home/coder/tmp/api__shushiato.parquet"
#df.to_parquet(output_path, engine="pyarrow", index=False) 

print(f"Сохранено {len(df)} строк в {output_path}")

Сохранено 4 строк в /home/coder/tmp/api__shushiato.parquet
